# 04 · Differential expression analysis (DEA)
**Input:** `data/processed/adata_annotated.h5ad`
**Output:** `data/processed/dea_results.csv`

Wilcoxon rank-sum test comparing tumor (HCC2) vs normal-adjacent (HCC1) tissue.
Threshold: padj < 0.05 and |log2FC| > 1.


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
adata = sc.read("../data/processed/adata_annotated.h5ad")
print(adata)

## Wilcoxon rank-sum test (tumor vs normal)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="sample", method="wilcoxon")
de_results = sc.get.rank_genes_groups_df(adata, group="tumor (HCC2)")

sig = de_results[
    (de_results["pvals_adj"] < 0.05) &
    (de_results["logfoldchanges"].abs() > 1)
].copy()
sig.columns = ["gene","log2FC","pvalue","adj_pvalue"]
sig["regulation"] = (sig["log2FC"] > 0).map({True:"up", False:"down"})

print(f"Total DEGs    : {len(sig)}")
print(f"Upregulated   : {(sig.regulation=='up').sum()}")
print(f"Downregulated : {(sig.regulation=='down').sum()}")
sig.head(10)

## Volcano plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 7))
colors = sig["regulation"].map({"up":"#D85A30","down":"#1D9E75"})
ax.scatter(sig["log2FC"], -np.log10(sig["adj_pvalue"]+1e-300),
           c=colors, alpha=0.7, s=20, linewidths=0)
ax.axvline(1,  color="#888", lw=0.8, ls="--")
ax.axvline(-1, color="#888", lw=0.8, ls="--")
ax.axhline(-np.log10(0.05), color="#888", lw=0.8, ls="--")

# Label top 10 by adjusted p-value
top10 = sig.nsmallest(10, "adj_pvalue")
for _, r in top10.iterrows():
    ax.text(r["log2FC"], -np.log10(r["adj_pvalue"]+1e-300)+0.3,
            r["gene"], fontsize=7, ha="center")

ax.set_xlabel("log2 fold change (tumor / normal)", fontsize=11)
ax.set_ylabel("-log10(adjusted p-value)", fontsize=11)
ax.set_title(f"DEA: HCC2 (tumor) vs HCC1 (normal)  —  {len(sig)} significant DEGs",
             fontsize=12)
import matplotlib.patches as mp
ax.legend(handles=[mp.Patch(facecolor="#D85A30",label="Up in tumor"),
                   mp.Patch(facecolor="#1D9E75",label="Down in tumor")], fontsize=9)
fig.savefig("../results/figures/volcano_plot.png", dpi=200, bbox_inches="tight")
plt.show()

## Export DEA results

In [ ]:
out = Path("../data/processed/dea_results.csv")
sig[["gene","log2FC","adj_pvalue","regulation"]].to_csv(out, index=False)
print(f"Saved: {out}  ({len(sig)} DEGs)")
print("Top 5 upregulated:")
print(sig[sig.regulation=="up"].nlargest(5,"log2FC")[["gene","log2FC","adj_pvalue"]])
print("\nTop 5 downregulated:")
print(sig[sig.regulation=="down"].nsmallest(5,"log2FC")[["gene","log2FC","adj_pvalue"]])